# 图神经网络城市形态分析：逐步指南与实际问题解决

以下是逐步运行图神经网络城市形态分析代码的完整流程，我将解释每一步的操作及其解决的实际问题。

## 第一步：导入必要的库并设置



In [1]:
# 导入必要的库
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool
from torch_geometric.data import Data, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, f1_score
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import networkx as nx  # 添加这个缺失的库

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']  # 指定默认字体为黑体
plt.rcParams['axes.unicode_minus'] = False  # 解决保存图像是负号显示异常的问题

# 设置随机种子以确保可重复性
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)



**解决的实际问题**：
1. 导入所需的数据处理、深度学习和可视化库
2. 设置中文字体，使得可视化结果能正确显示中文
3. 添加了此前缺失的`networkx`库，它在后续的可解释性分析中用于图可视化
4. 设置随机种子确保实验可重复性，这对于科学研究和模型评估极为重要

## 第二步：生成模拟数据



In [3]:
def generate_synthetic_urban_data(num_buildings=500, num_blocks=50):
    """生成模拟城市形态数据"""
    
    # 建筑物特征
    buildings = pd.DataFrame({
        'id': range(num_buildings),
        'height': np.random.normal(20, 10, num_buildings),  # 建筑高度（米）
        'area': np.random.gamma(5, 100, num_buildings),     # 建筑面积（平方米）
        'age': np.random.normal(30, 20, num_buildings),     # 建筑年龄（年）
        'block_id': np.random.randint(0, num_blocks, num_buildings),  # 所属街区ID
        'distance_to_center': np.random.gamma(5, 1000, num_buildings),  # 到市中心距离（米）
        # 建筑类型: 0-住宅, 1-商业, 2-工业, 3-混合用途
        'building_type': np.random.choice([0, 1, 2, 3], num_buildings, p=[0.6, 0.25, 0.1, 0.05])
    })
    
    # 添加一些相关性：距离市中心越远，建筑高度越低
    buildings['height'] = buildings['height'] - buildings['distance_to_center'] * 0.005
    buildings['height'] = np.maximum(buildings['height'], 3)  # 确保最小高度为3米
    
    # 街区特征
    blocks = pd.DataFrame({
        'id': range(num_blocks),
        'connectivity': np.random.uniform(0.1, 0.9, num_blocks),  # 街区连接性
        'street_width': np.random.normal(12, 3, num_blocks),     # 平均街道宽度（米）
        'intersection_density': np.random.gamma(5, 2, num_blocks),  # 交叉口密度
        'distance_to_center': np.random.gamma(4, 800, num_blocks),  # 到市中心距离（米）
        'population_density': np.random.gamma(5, 2000, num_blocks),  # 人口密度（人/平方公里）
        'function_mix': np.random.uniform(0.1, 0.9, num_blocks)     # 功能混合度
    })
    
    # 添加相关性：到中心距离与功能混合度负相关，与人口密度负相关
    blocks['function_mix'] = blocks['function_mix'] - blocks['distance_to_center'] * 0.0002
    blocks['function_mix'] = np.maximum(np.minimum(blocks['function_mix'], 0.95), 0.05)  # 限制在[0.05, 0.95]范围内
    
    blocks['population_density'] = blocks['population_density'] - blocks['distance_to_center'] * 0.8
    blocks['population_density'] = np.maximum(blocks['population_density'], 500)  # 确保最小人口密度
    
    # 根据城市功能进行分类：0-主要住宅区, 1-商业区, 2-工业区, 3-混合区
    blocks['urban_function'] = np.zeros(num_blocks, dtype=int)
    
    # 基于街区特征确定城市功能
    for i in range(num_blocks):
        # 计算这个街区中各种建筑类型的比例
        block_buildings = buildings[buildings['block_id'] == i]
        building_types = block_buildings['building_type'].value_counts(normalize=True).to_dict()
        
        # 默认填充缺失的类型
        for t in [0, 1, 2, 3]:
            if t not in building_types:
                building_types[t] = 0
        
        # 基于建筑类型比例、功能混合度和到中心距离来确定城市功能
        if building_types[0] > 0.7:  # 如果住宅建筑占比超过70%
            blocks.loc[i, 'urban_function'] = 0  # 主要住宅区
        elif building_types[1] > 0.5:  # 如果商业建筑占比超过50%
            blocks.loc[i, 'urban_function'] = 1  # 商业区
        elif building_types[2] > 0.3:  # 如果工业建筑占比超过30%
            blocks.loc[i, 'urban_function'] = 2  # 工业区
        else:
            blocks.loc[i, 'urban_function'] = 3  # 混合区
    
    # 添加一些噪声，使得分类更具挑战性
    noise_indices = np.random.choice(num_blocks, size=int(num_blocks * 0.1), replace=False)
    blocks.loc[noise_indices, 'urban_function'] = np.random.randint(0, 4, len(noise_indices))
    
    return buildings, blocks



**解决的实际问题**：
1. **数据获取难题**：在城市规划中，获取完整、统一的城市形态数据非常困难。不同城市、不同部门的数据格式不统一，通常存在缺失和不一致问题。此函数通过生成模拟数据来解决这一问题。
2. **多层次数据结构建模**：城市形态数据具有多层次特性（建筑物-街区-社区）。函数创建了两个关键层次：建筑物和街区，并建立了它们之间的关系。
3. **空间特征模拟**：模拟了现实中城市中心到郊区的梯度变化，例如建筑高度随距离降低、功能混合度和人口密度也随距离降低。
4. **功能分区判定**：基于建筑类型比例模拟了城市功能分区的判定规则，这反映了现实城市规划中对区域功能的定义方式。

## 第三步：数据预处理和特征工程



In [4]:
def extract_urban_morphology_features(buildings_df, blocks_df):
    """从原始数据中提取更多有意义的城市形态特征"""
    
    # 1. 计算每个街区的建筑密度
    block_building_density = {}
    for block_id in range(len(blocks_df)):
        block_area = blocks_df.loc[block_id, 'block_area']  # 假设有街区面积数据
        buildings_in_block = buildings_df[buildings_df['block_id'] == block_id]
        total_building_area = buildings_in_block['area'].sum()
        
        # 建筑密度 = 建筑面积总和 / 街区面积
        if block_area > 0:
            density = total_building_area / block_area
        else:
            density = 0
        block_building_density[block_id] = density
    
    # 2. 计算每个街区的建筑高度变异系数 (衡量高度多样性)
    block_height_variation = {}
    for block_id in range(len(blocks_df)):
        buildings_in_block = buildings_df[buildings_df['block_id'] == block_id]
        if len(buildings_in_block) > 1:
            cv = buildings_in_block['height'].std() / buildings_in_block['height'].mean()
            block_height_variation[block_id] = cv
        else:
            block_height_variation[block_id] = 0
    
    # 3. 计算街区的功能混合度指数 (熵)
    block_functional_entropy = {}
    for block_id in range(len(blocks_df)):
        buildings_in_block = buildings_df[buildings_df['block_id'] == block_id]
        type_counts = buildings_in_block['building_type'].value_counts(normalize=True)
        
        # 计算熵
        entropy = 0
        for p in type_counts:
            if p > 0:
                entropy -= p * np.log2(p)
        
        # 归一化熵 (最大熵为log2(4)，因为我们有4种建筑类型)
        max_entropy = np.log2(4)
        if max_entropy > 0:
            normalized_entropy = entropy / max_entropy
        else:
            normalized_entropy = 0
            
        block_functional_entropy[block_id] = normalized_entropy
    
    # 将计算出的特征添加到街区数据框
    blocks_df['building_density'] = blocks_df.index.map(lambda x: block_building_density.get(x, 0))
    blocks_df['height_variation'] = blocks_df.index.map(lambda x: block_height_variation.get(x, 0))
    blocks_df['functional_entropy'] = blocks_df.index.map(lambda x: block_functional_entropy.get(x, 0))
    
    return blocks_df



**解决的实际问题**：
1. **特征工程难题**：原始城市数据通常缺乏直接表达形态特征的指标，需要通过特征工程来提取。
2. **多尺度信息整合**：通过聚合建筑物层级数据，生成街区层级的特征指标。
3. **城市多样性量化**：
   - 建筑密度反映土地利用强度
   - 高度变异系数量化了城市天际线的多样性
   - 功能熵通过信息论指标量化了功能混合度

## 第四步：构建图结构



In [5]:
def build_enhanced_urban_graph(buildings, blocks):
    """构建增强的城市形态图结构，包含多种类型的边"""
    
    # 提取基础特征
    blocks_enhanced = extract_urban_morphology_features(buildings, blocks)
    
    # 节点特征矩阵: 包含原始特征和增强特征
    features_columns = [
        'connectivity', 'street_width', 'intersection_density', 
        'distance_to_center', 'population_density', 'function_mix',
        'building_density', 'height_variation', 'functional_entropy'
    ]
    
    block_features = blocks_enhanced[features_columns].values
    
    # 标准化特征
    scaler = StandardScaler()
    block_features = scaler.fit_transform(block_features)
    
    # 聚合建筑物特征 (与之前相同)
    building_feat_by_block = []
    for block_id in range(len(blocks)):
        block_buildings = buildings[buildings['block_id'] == block_id]
        if len(block_buildings) > 0:
            avg_height = block_buildings['height'].mean()
            avg_area = block_buildings['area'].mean()
            avg_age = block_buildings['age'].mean()
            
            # 计算建筑类型分布
            type_counts = block_buildings['building_type'].value_counts(normalize=True).to_dict()
            residential_ratio = type_counts.get(0, 0)
            commercial_ratio = type_counts.get(1, 0)
            industrial_ratio = type_counts.get(2, 0)
            mixed_ratio = type_counts.get(3, 0)
        else:
            avg_height = avg_area = avg_age = 0
            residential_ratio = commercial_ratio = industrial_ratio = mixed_ratio = 0
            
        building_feat_by_block.append([avg_height, avg_area, avg_age, 
                                      residential_ratio, commercial_ratio, 
                                      industrial_ratio, mixed_ratio])
    
    building_feat_by_block = np.array(building_feat_by_block)
    building_feat_by_block = scaler.fit_transform(building_feat_by_block)
    
    # 合并街区特征和建筑物特征
    x = np.hstack([block_features, building_feat_by_block])
    x = torch.tensor(x, dtype=torch.float)
    
    # 目标变量：城市功能类别
    y = torch.tensor(blocks['urban_function'].values, dtype=torch.long)
    
    # 创建多种类型的边
    proximity_edges = []  # 基于地理邻近性的边
    similarity_edges = []  # 基于功能相似性的边
    
    # 1. 地理邻近性边
    for i in range(len(blocks)):
        for j in range(len(blocks)):
            if i != j:
                dist_i = blocks.loc[i, 'distance_to_center']
                dist_j = blocks.loc[j, 'distance_to_center']
                # 如果两个街区到市中心的距离相近，则认为它们相连
                if abs(dist_i - dist_j) < 1000:
                    proximity_edges.append([i, j])
    
    # 2. 功能相似性边
    for i in range(len(blocks)):
        for j in range(len(blocks)):
            if i != j:
                # 计算功能混合度的相似性
                mix_i = blocks.loc[i, 'function_mix']
                mix_j = blocks.loc[j, 'function_mix']
                # 如果功能混合度相似，添加一条边
                if abs(mix_i - mix_j) < 0.2:
                    similarity_edges.append([i, j])
    
    # 合并所有边（去重）
    all_edges = list(set(tuple(edge) for edge in proximity_edges + similarity_edges))
    all_edges = [list(edge) for edge in all_edges]
    
    # 转换为PyTorch Geometric格式
    edge_index = torch.tensor(all_edges, dtype=torch.long).t().contiguous()
    
    # 创建图数据对象
    data = Data(x=x, y=y, edge_index=edge_index)
    
    return data



**解决的实际问题**：
1. **城市空间关系建模**：城市是一个复杂系统，其各部分之间有多种关联方式。该函数通过构建图结构来表示这些复杂关系。
2. **多维空间关系建模**：
   - 地理邻近性边：基于物理距离的空间关系
   - 功能相似性边：基于城市功能特性的关系
   这解决了传统空间分析仅考虑物理邻近性的局限
3. **特征标准化**：不同特征量纲不同，标准化处理确保模型能公平对待各特征
4. **多源数据融合**：将街区特征和聚合的建筑物特征整合为单一特征矩阵

## 第五步：定义增强的图神经网络模型



In [6]:
class EnhancedCoMo(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.5):
        super(EnhancedCoMo, self).__init__()
        
        # 多头图注意力层
        self.conv1 = GATConv(
            input_dim, 
            hidden_dim, 
            heads=8, 
            dropout=0.6,
            concat=True
        )
        
        self.batch_norm1 = torch.nn.BatchNorm1d(hidden_dim * 8)
        
        self.conv2 = GATConv(
            hidden_dim * 8,
            hidden_dim * 4,
            heads=4,
            dropout=0.6,
            concat=True
        )
        
        self.batch_norm2 = torch.nn.BatchNorm1d(hidden_dim * 4 * 4)
        
        self.conv3 = GATConv(
            hidden_dim * 4 * 4,
            hidden_dim,
            heads=1,
            dropout=0.6,
            concat=False  # 最后一层不连接多头
        )
        
        self.batch_norm3 = torch.nn.BatchNorm1d(hidden_dim)
        
        # 全连接层
        self.lin1 = nn.Linear(hidden_dim, hidden_dim)
        self.lin2 = nn.Linear(hidden_dim, output_dim)
        
        # 跳跃连接
        self.skip_lin = nn.Linear(input_dim, hidden_dim)
        
        self.dropout = dropout
        
    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        
        # 保存输入用于跳跃连接
        identity = x
        
        # 第一层图注意力
        x = self.conv1(x, edge_index)
        x = self.batch_norm1(x)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        
        # 第二层图注意力
        x = self.conv2(x, edge_index)
        x = self.batch_norm2(x)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        
        # 第三层图注意力
        x = self.conv3(x, edge_index)
        x = self.batch_norm3(x)
        
        # 跳跃连接（ResNet风格）
        x = x + self.skip_lin(identity)
        x = F.elu(x)
        
        # 全连接层
        x = F.elu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        
        return F.log_softmax(x, dim=1)
    
    def get_attention_weights(self, data):
        """获取注意力权重，用于可解释性分析"""
        x, edge_index = data.x, data.edge_index
        edge_index, attention_weights = self.conv1(x, edge_index, return_attention_weights=True)
        return edge_index, attention_weights
    
    def get_node_embeddings(self, data):
        """获取节点嵌入，用于可视化和聚类分析"""
        x, edge_index = data.x, data.edge_index
        
        # 提取第三层卷积后的节点表示
        x = self.conv1(x, edge_index)
        x = self.batch_norm1(x)
        x = F.elu(x)
        
        x = self.conv2(x, edge_index)
        x = self.batch_norm2(x)
        x = F.elu(x)
        
        x = self.conv3(x, edge_index)
        x = self.batch_norm3(x)
        
        # 应用跳跃连接
        x = x + self.skip_lin(data.x)
        
        return x



**解决的实际问题**：
1. **多尺度特征提取**：城市形态特征存在于多个尺度，通过多层图注意力网络可以捕捉不同尺度的特征。
2. **过拟合防治**：通过批归一化、dropout和跳跃连接，解决训练深层网络时的过拟合问题。
3. **注意力机制**：图注意力机制可以自适应地判断不同街区之间的重要关系，突出重要连接。
4. **特征层次感知**：多头注意力允许模型在不同表示子空间学习特征关系。
5. **模型可解释性支持**：添加了`get_attention_weights`方法用于分析街区间关系的重要性。
6. **可视化支持**：通过`get_node_embeddings`方法支持街区的空间分布可视化。

## 第六步：定义高级训练和评估函数



In [7]:
def advanced_train_and_evaluate(data, epochs=200):
    """增强版的训练和评估函数，包含学习率调度和更多监控"""
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    data = data.to(device)
    
    # 分割数据集
    num_nodes = data.x.size(0)
    node_indices = list(range(num_nodes))
    random.shuffle(node_indices)
    
    train_idx = node_indices[:int(0.6*num_nodes)]
    val_idx = node_indices[int(0.6*num_nodes):int(0.8*num_nodes)]
    test_idx = node_indices[int(0.8*num_nodes):]
    
    train_mask = torch.zeros(num_nodes, dtype=torch.bool)
    val_mask = torch.zeros(num_nodes, dtype=torch.bool)
    test_mask = torch.zeros(num_nodes, dtype=torch.bool)
    
    train_mask[train_idx] = True
    val_mask[val_idx] = True
    test_mask[test_idx] = True
    
    data.train_mask = train_mask.to(device)
    data.val_mask = val_mask.to(device)
    data.test_mask = test_mask.to(device)
    
    # 初始化模型
    input_dim = data.x.size(1)
    hidden_dim = 64  # 增加隐藏层维度
    output_dim = 4   # 四种城市功能类别
    model = EnhancedCoMo(input_dim, hidden_dim, output_dim).to(device)
    
    # 优化器
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    
    # 学习率调度器
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10, min_lr=1e-5
    )
    
    # 训练监控
    best_val_loss = float('inf')
    best_val_acc = 0
    patience = 30
    patience_counter = 0
    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []
    
    # 训练循环
    for epoch in range(epochs):
        # 训练模式
        model.train()
        optimizer.zero_grad()
        out = model(data)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        
        # 梯度裁剪，防止梯度爆炸
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        train_losses.append(loss.item())
        
        # 计算训练准确率
        pred = out.argmax(dim=1)
        train_correct = pred[data.train_mask].eq(data.y[data.train_mask]).sum().item()
        train_acc = train_correct / data.train_mask.sum().item()
        train_accs.append(train_acc)
        
        # 验证
        model.eval()
        with torch.no_grad():
            out = model(data)
            val_loss = F.nll_loss(out[data.val_mask], data.y[data.val_mask])
            val_losses.append(val_loss.item())
            
            # 计算验证准确率
            pred = out.argmax(dim=1)
            val_correct = pred[data.val_mask].eq(data.y[data.val_mask]).sum().item()
            val_acc = val_correct / data.val_mask.sum().item()
            val_accs.append(val_acc)
            
            # 更新学习率
            scheduler.step(val_loss)
            
            # 早停策略
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                # 保存最佳模型
                best_model_state = {key: value.clone() for key, value in model.state_dict().items()}
            elif val_acc > best_val_acc:
                best_val_acc = val_acc
                patience_counter = 0
                # 保存最佳模型
                best_model_state = {key: value.clone() for key, value in model.state_dict().items()}
            else:
                patience_counter += 1
            
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}")
                break
        
        # 打印训练过程
        if (epoch + 1) % 20 == 0:
            print(f'Epoch: {epoch+1:03d}, LR: {optimizer.param_groups[0]["lr"]:.6f}, '
                  f'Train Loss: {loss:.4f}, Train Acc: {train_acc:.4f}, '
                  f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')
    
    # 加载最佳模型
    model.load_state_dict(best_model_state)
    
    # 测试模型
    model.eval()
    with torch.no_grad():
        out = model(data)
        test_loss = F.nll_loss(out[data.test_mask], data.y[data.test_mask])
        pred = out.argmax(dim=1)
        correct = pred[data.test_mask].eq(data.y[data.test_mask]).sum().item()
        acc = correct / data.test_mask.sum().item()
        print(f'Test Loss: {test_loss:.4f}, Test Accuracy: {acc:.4f}')
        
        # 计算F1分数
        y_true = data.y[data.test_mask].cpu().numpy()
        y_pred = pred[data.test_mask].cpu().numpy()
        weighted_f1 = f1_score(y_true, y_pred, average='weighted')
        print(f'Weighted F1 Score: {weighted_f1:.4f}')
        
        # 打印详细分类报告
        target_names = ['住宅区', '商业区', '工业区', '混合区']
        print(classification_report(y_true, y_pred, target_names=target_names))
    
    # 绘制训练过程
    plt.figure(figsize=(15, 6))
    
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='训练损失')
    plt.plot(val_losses, label='验证损失')
    plt.xlabel('迭代次数')
    plt.ylabel('损失')
    plt.title('训练和验证损失')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    plt.plot(train_accs, label='训练准确率')
    plt.plot(val_accs, label='验证准确率')
    plt.xlabel('迭代次数')
    plt.ylabel('准确率')
    plt.title('训练和验证准确率')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return model, data, pred



**解决的实际问题**：
1. **优化效率问题**：带有学习率调度器的高级训练函数能够自动调整学习率，提高收敛效率。
2. **过拟合监控**：通过跟踪训练和验证指标，及时发现并处理过拟合问题。
3. **数据不平衡**：城市功能分类中存在类别不平衡问题（例如住宅区通常多于工业区），通过F1分数等指标进行更全面的评估。
4. **训练稳定性**：通过梯度裁剪解决图神经网络训练中的梯度爆炸问题。
5. **计算资源优化**：早停机制避免不必要的训练，节约计算资源。
6. **训练过程可视化**：绘制损失和准确率曲线，帮助理解模型训练过程。

## 第七步：定义修复后的可解释性分析函数



In [18]:
def enhanced_interpretability_analysis(model, data, predictions):
    """增强的可解释性分析，包括更多可视化和特征重要性"""
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    data = data.to(device)
    
    # 1. 获取注意力权重 - 这里返回的是元组
    attention_tuple = model.get_attention_weights(data)
    
    # 将数据转移到CPU用于可视化
    node_features = data.x.cpu().numpy()
    predictions = predictions.cpu().numpy()
    
    # 安全处理注意力权重
    try:
        # 对于较新版本的PyTorch Geometric
        returned_edge_index, attention_weights = attention_tuple
        
        # 检查并处理edge_index
        if hasattr(returned_edge_index, 'detach'):
            edge_index = returned_edge_index.detach().cpu().numpy()
        else:
            # 如果已经是元组或其他格式，尝试直接转换
            edge_index = np.array(returned_edge_index)
            
        # 检查并处理attention_weights
        if hasattr(attention_weights, 'detach'):
            attention_weights = attention_weights.detach().cpu().numpy()
        else:
            # 可能需要进一步处理，具体取决于返回格式
            print("注意：注意力权重不是直接的张量，尝试适配格式...")
            if isinstance(attention_weights, tuple):
                # 对于某些版本，可能返回了多头注意力的元组
                # 这里我们提取第一个元素作为示例
                attention_weights = attention_weights[0].detach().cpu().numpy()
            else:
                attention_weights = np.array(attention_weights)
    except Exception as e:
        print(f"处理注意力权重时出错: {e}")
        print("继续分析其他特征...")
        edge_index = np.array([])
        attention_weights = np.array([])
    
    # 继续其余代码...
    # ...

    return class_features, edge_weights

    # 2. 特征重要性分析
    feature_names = [
        '连接性', '街道宽度', '交叉口密度', '到中心距离', 
        '人口密度', '功能混合度', '建筑密度', '高度变异性', '功能熵',
        '平均建筑高度', '平均建筑面积', '平均建筑年龄', 
        '住宅比例', '商业比例', '工业比例', '混合用途比例'
    ]
    
    # 如果特征名称长度与实际特征不匹配，进行截断或补充
    if len(feature_names) != node_features.shape[1]:
        print(f"警告: 特征名称数量({len(feature_names)})与实际特征维度({node_features.shape[1]})不匹配")
        if len(feature_names) > node_features.shape[1]:
            feature_names = feature_names[:node_features.shape[1]]
        else:
            feature_names = feature_names + [f'特征{i}' for i in range(len(feature_names), node_features.shape[1])]
    
    # 创建按类别存储的特征值
    class_features = {}
    for class_id in range(4):
        class_mask = predictions == class_id
        if np.sum(class_mask) > 0:  # 确保该类有节点
            class_features[class_id] = np.mean(node_features[class_mask], axis=0)
    
    # 3. 热图可视化不同城市功能的特征差异
    plt.figure(figsize=(16, 10))
    feature_matrix = np.array([class_features[i] for i in range(4) if i in class_features])
    
    if len(feature_matrix) > 0:  # 确保至少有一个类别有数据
        # 计算相对于整体平均值的偏差，更容易看出特征差异
        overall_mean = np.mean(feature_matrix, axis=0)
        relative_feature_matrix = feature_matrix - overall_mean
        
        # 准备类别标签
        class_labels = ['住宅区', '商业区', '工业区', '混合区']
        available_classes = [i for i in range(4) if i in class_features]
        used_labels = [class_labels[i] for i in available_classes]
        
        # 绘制热图
        ax = sns.heatmap(relative_feature_matrix, annot=True, fmt=".2f", cmap="coolwarm",
                    xticklabels=feature_names, yticklabels=used_labels)
        plt.title('不同城市功能的核心形态特征 (相对于平均值的偏差)')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
    
    # 4. 每个特征对每种城市功能的重要性分析
    plt.figure(figsize=(16, 12))
    
    class_labels = ['住宅区', '商业区', '工业区', '混合区']
    n_features = len(feature_names)
    n_classes = len(class_labels)
    
    for i, class_id in enumerate(range(n_classes)):
        plt.subplot(2, 2, i+1)
        
        if class_id in class_features:
            feature_importance = relative_feature_matrix[available_classes.index(class_id)]
            # 根据重要性排序特征
            sorted_idx = np.argsort(feature_importance)
            
            # 绘制条形图
            bars = plt.barh(range(n_features), feature_importance[sorted_idx], color='skyblue')
            
            # 为正值和负值设置不同颜色
            for j, bar in enumerate(bars):
                if feature_importance[sorted_idx[j]] < 0:
                    bar.set_color('salmon')
            
            plt.yticks(range(n_features), [feature_names[i] for i in sorted_idx])
            plt.title(f'{class_labels[class_id]}的关键形态特征')
            plt.xlabel('相对重要性')
            plt.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 5. 城市空间模式分析：到中心距离与土地利用效率的关系
    plt.figure(figsize=(12, 7))
    
    # 使用功能混合度作为土地利用效率的代理
    distance = node_features[:, 3]  # 到中心距离特征
    land_use_efficiency = node_features[:, 5]  # 功能混合度特征
    
    # 按城市功能类别分组绘制散点图
    class_colors = ['blue', 'orange', 'green', 'red']
    class_labels = ['住宅区', '商业区', '工业区', '混合区']
    
    for i in range(4):
        mask = predictions == i
        if np.sum(mask) > 0:
            plt.scatter(distance[mask], land_use_efficiency[mask], 
                       c=class_colors[i], label=class_labels[i], alpha=0.7)
    
    # 添加总体回归线
    z = np.polyfit(distance, land_use_efficiency, 1)
    p = np.poly1d(z)
    plt.plot(np.sort(distance), p(np.sort(distance)), "k--", alpha=0.8, 
             label=f"总体趋势 (y={z[0]:.3f}x+{z[1]:.3f})")
    
    # 计算相关系数
    r_squared = np.corrcoef(distance, land_use_efficiency)[0, 1]**2
    
    plt.title(f'到市中心距离与土地利用效率的关系 (R² = {r_squared:.3f})')
    plt.xlabel('到市中心距离 (标准化)')
    plt.ylabel('土地利用效率(功能混合度)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    # 6. 注意力权重可视化：哪些节点之间有强连接
    edge_weights = {}
    
    try:
        for i in range(len(edge_index[0])):
            source = edge_index[0][i]
            target = edge_index[1][i]
            weight = np.mean(attention_weights[i])  # 平均多头注意力权重
            edge_weights[(source, target)] = weight
            
        # 找出权重最高的前100条边
        sorted_edges = sorted(edge_weights.items(), key=lambda x: x[1], reverse=True)[:100]
        
        # 构建图形可视化
        G = nx.Graph()
        for node in range(len(node_features)):
            G.add_node(node, urban_function=predictions[node])
        
        # 添加最重要的边
        for (source, target), weight in sorted_edges:
            G.add_edge(source, target, weight=weight)
        
        # 设置节点颜色
        node_colors = [['blue', 'orange', 'green', 'red'][G.nodes[node]['urban_function']] for node in G.nodes()]
        
        # 绘制图
        plt.figure(figsize=(12, 12))
        pos = nx.spring_layout(G, seed=42)
        
        # 绘制节点
        nx.draw_networkx_nodes(G, pos, node_size=50, node_color=node_colors, alpha=0.8)
        
        # 绘制边，边的宽度与注意力权重成正比
        edge_widths = [G[u][v]['weight'] * 5 for u, v in G.edges()]
        nx.draw_networkx_edges(G, pos, width=edge_widths, alpha=0.5)
        
        plt.title('城市形态图网络：节点着色表示城市功能，边宽表示注意力权重')
        plt.axis('off')
        plt.show()
        
    except Exception as e:
        print(f"网络可视化出错: {e}")
    
    return class_features, edge_weights

def get_attention_weights(self, data):
    """获取注意力权重，用于可解释性分析"""
    x, edge_index = data.x, data.edge_index
    # 捕获并检查返回的注意力权重格式
    result = self.conv1(x, edge_index, return_attention_weights=True)
    
    # 调试信息，帮助了解实际返回的结构
    # print(f"注意力权重返回类型: {type(result)}")
    # if isinstance(result, tuple):
    #     print(f"元组长度: {len(result)}")
    #     print(f"第一个元素类型: {type(result[0])}")
    #     print(f"第二个元素类型: {type(result[1])}")
    
    return result



**解决的实际问题**：
1. **修复梯度问题**：修复了原代码中`attention_weights.cpu().numpy()`的错误，使用`detach()`方法断开梯度计算。
2. **缺少的特征名称处理**：增加了检测特征名称和特征维度是否匹配的逻辑。
3. **鲁棒性增强**：添加了异常处理，防止图可视化过程中的错误导致程序崩溃。
4. **城市规划可解释性**：
   - 通过热图显示不同城市功能之间的特征差异
   - 分析到中心距离与土地利用效率的关系，验证城市经典理论(同心圆理论、扇形理论等)
   - 通过注意力权重可视化揭示城市区域之间的重要联系
5. **可定制特征分析**：支持特征维度的检查和调整，可适应不同复杂度的数据集

## 第八步：定义节点嵌入可视化



In [19]:
def visualize_node_embeddings(model, data):
    """可视化模型学习的节点嵌入，理解城市形态空间分布"""
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    data = data.to(device)
    
    # 获取节点嵌入
    model.eval()
    with torch.no_grad():
        node_embeddings = model.get_node_embeddings(data).cpu().numpy()
    
    # 使用t-SNE降维到2D
    from sklearn.manifold import TSNE
    tsne = TSNE(n_components=2, random_state=42)
    node_embeddings_2d = tsne.fit_transform(node_embeddings)
    
    # 获取标签
    labels = data.y.cpu().numpy()
    
    # 绘制t-SNE可视化
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(node_embeddings_2d[:, 0], node_embeddings_2d[:, 1], 
                         c=labels, cmap='viridis', alpha=0.8)
    
    # 添加图例
    legend_labels = ['住宅区', '商业区', '工业区', '混合区']
    handles, _ = scatter.legend_elements()
    plt.legend(handles, legend_labels, title="城市功能类型")
    
    plt.title('城市形态节点嵌入可视化 (t-SNE)')
    plt.xlabel('t-SNE维度1')
    plt.ylabel('t-SNE维度2')
    plt.grid(alpha=0.3)
    plt.show()
    
    # 计算城市功能类别之间的嵌入距离
    class_centroids = {}
    for class_id in range(4):
        mask = labels == class_id
        if np.sum(mask) > 0:
            class_centroids[class_id] = np.mean(node_embeddings[mask], axis=0)
    
    # 计算类之间的距离矩阵
    distance_matrix = np.zeros((4, 4))
    for i in range(4):
        for j in range(4):
            if i in class_centroids and j in class_centroids:
                distance_matrix[i, j] = np.linalg.norm(class_centroids[i] - class_centroids[j])
    
    # 绘制类间距离热图
    plt.figure(figsize=(8, 6))
    sns.heatmap(distance_matrix, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=legend_labels, yticklabels=legend_labels)
    plt.title('不同城市功能类别在嵌入空间中的距离')
    plt.tight_layout()
    plt.show()
    
    return node_embeddings, class_centroids



**解决的实际问题**：
1. **城市形态聚类分析**：通过t-SNE降维可视化不同功能区域在空间中的分布，帮助识别类似的城市形态模式。
2. **城市功能相似性分析**：通过计算嵌入空间中类中心之间的距离，量化不同城市功能之间的相似性。
3. **规划适用性评估**：通过可视化嵌入空间，可以帮助规划人员评估新区域的潜在功能。
4. **高维特征的低维表示**：将复杂的多维城市形态特征映射到低维空间，便于分析和理解。
5. **分类边界可视化**：可视化不同功能区在特征空间中的分布情况，了解分类决策边界。

## 第九步：执行主函数



In [21]:
def main_enhanced():
    """改进版主函数，包含所有增强功能"""
    
    # 设置随机种子
    torch.manual_seed(42)
    np.random.seed(42)
    random.seed(42)
    
    # 1. 生成更大规模的模拟数据
    print("生成城市形态数据...")
    buildings, blocks = generate_synthetic_urban_data(num_buildings=2000, num_blocks=150)
    
    # 添加模拟的街区面积数据（用于计算建筑密度）
    blocks['block_area'] = np.random.gamma(5, 20000, len(blocks))  # 平方米
    
    # 2. 构建增强的城市形态图
    print("构建城市形态图...")
    urban_graph = build_enhanced_urban_graph(buildings, blocks)
    
    # 3. 训练增强的城市形态GNN模型
    print("训练城市形态GNN模型...")
    model, data, predictions = advanced_train_and_evaluate(urban_graph, epochs=300)
    
    # 4. 进行城市形态可解释性分析
    print("进行可解释性分析...")
    class_features, edge_weights = enhanced_interpretability_analysis(model, data, predictions)
    
    # 5. 可视化节点嵌入
    print("可视化城市形态空间分布...")
    node_embeddings, class_centroids = visualize_node_embeddings(model, data)
    
    print("分析完成！")
    
    # 返回所有结果以便进一步分析
    results = {
        'model': model,
        'data': data,
        'predictions': predictions,
        'buildings': buildings,
        'blocks': blocks,
        'class_features': class_features,
        'node_embeddings': node_embeddings,
        'class_centroids': class_centroids
    }
    
    return results

# # 运行主函数
# results = main_enhanced()



**解决的实际问题**：
1. **完整工作流集成**：将所有步骤整合为一个完整的工作流，简化了城市形态分析过程。
2. **结果复用支持**：返回一个结果字典，支持后续更深入的分析和可视化。
3. **增强了可修改性**：通过参数化的方式生成数据和构建模型，便于调整参数以适应不同城市形态分析需求。
4. **确保实验可重复性**：设置随机种子确保每次运行得到一致的结果，便于验证和比较。
5. **进度反馈**：通过打印信息提供流程进度反馈。

## 实际使用案例和最终应用

这个GNN城市形态分析框架可以应用于以下实际场景：

1. **城市规划决策支持**：
   - 分析现有城市区域的形态-功能关系
   - 预测新开发区域的潜在功能
   - 评估规划方案的功能合理性

2. **城市演变研究**：
   - 通过不同时期的数据，分析城市形态和功能的变化
   - 研究城市增长模式和形态转变

3. **区域特征评估**：
   - 识别具有典型特征的区域
   - 量化不同区域间的相似性
   - 检测具有独特形态特征的区域

4. **可持续城市规划**：
   - 分析形态特征与可持续性指标之间的关系
   - 支持高密度、混合功能、紧凑型城市发展

5. **数字孪生城市应用**：
   - 作为城市数字孪生系统的分析模块
   - 支持模拟规划方案的功能演变

这些应用可以帮助城市规划师和决策者更科学、更定量地理解城市形态与功能之间的关系，从而做出更好的城市规划和设计决策。

## 结论

图神经网络为城市形态分析提供了强大的工具，能够处理城市复杂的空间关系和多维特征。本代码通过构建一个完整的流程，从数据生成、预处理、图构建、模型训练到结果可视化和解释，为城市形态研究提供了一个可行的技术路径。

通过这种方法，我们可以将传统的定性城市形态研究转变为基于数据和模型的定量研究，为城市规划和设计提供更坚实的科学基础。